# Model Training
Train 3 models, compare them, and save the best one.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from feature_engineering import get_feature_columns

os.makedirs('model', exist_ok=True)

df = pd.read_csv('data/featured_data.csv')
print('Rows:', len(df))
df.head()

## Prepare Features and Target

In [ ]:
feature_cols = get_feature_columns()

# Add any missing columns as 0
for col in feature_cols:
    if col not in df.columns:
        df[col] = 0

X = df[feature_cols]
y = df['Time_taken(min)']

# Drop rows with missing values
data = pd.concat([X, y], axis=1).dropna()
X = data[feature_cols]
y = data['Time_taken(min)']

print('Total rows ready:', len(X))
print('Features used   :', len(feature_cols))

## Train / Test Split (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))

## Model 1 — Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
preds_lr = lr.predict(X_test)

print('Linear Regression')
print('  MAE  :', round(mean_absolute_error(y_test, preds_lr), 2), 'minutes')
print('  RMSE :', round(np.sqrt(mean_squared_error(y_test, preds_lr)), 2))
print('  R2   :', round(r2_score(y_test, preds_lr), 4))

## Model 2 — Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

print('Random Forest')
print('  MAE  :', round(mean_absolute_error(y_test, preds_rf), 2), 'minutes')
print('  RMSE :', round(np.sqrt(mean_squared_error(y_test, preds_rf)), 2))
print('  R2   :', round(r2_score(y_test, preds_rf), 4))

## Model 3 — XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                   random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)

print('XGBoost')
print('  MAE  :', round(mean_absolute_error(y_test, preds_xgb), 2), 'minutes')
print('  RMSE :', round(np.sqrt(mean_squared_error(y_test, preds_xgb)), 2))
print('  R2   :', round(r2_score(y_test, preds_xgb), 4))

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE'  : [round(mean_absolute_error(y_test, preds_lr), 2),
              round(mean_absolute_error(y_test, preds_rf), 2),
              round(mean_absolute_error(y_test, preds_xgb), 2)],
    'R2'   : [round(r2_score(y_test, preds_lr), 4),
              round(r2_score(y_test, preds_rf), 4),
              round(r2_score(y_test, preds_xgb), 4)],
})
comparison = comparison.sort_values('MAE').reset_index(drop=True)
print('Best Model:', comparison.iloc[0]['Model'])
comparison

## Feature Importance (XGBoost)

In [ ]:
import matplotlib.pyplot as plt

imp = pd.DataFrame({'Feature': feature_cols, 'Importance': xgb.feature_importances_})
imp = imp.sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
plt.barh(imp['Feature'][::-1], imp['Importance'][::-1], color='steelblue')
plt.title('Top 10 Features — XGBoost')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
imp

## Save the Best Model

In [ ]:
with open('model/best_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

with open('model/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('model/model_info.pkl', 'wb') as f:
    pickle.dump({
        'name': 'XGBoost',
        'mae' : round(mean_absolute_error(y_test, preds_xgb), 2),
        'rmse': round(np.sqrt(mean_squared_error(y_test, preds_xgb)), 2),
        'r2'  : round(r2_score(y_test, preds_xgb), 4)
    }, f)

imp.to_csv('model/feature_importance.csv', index=False)
print('Model saved to model/best_model.pkl')